In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [5]:
ENTREPOT_PATH = '/home/administrateur/Bureau/Datagrosyst/data_entrepot_outils/'
donnees = {}

def import_df(df_name, path_data, sep, index_col=None):
    donnees[df_name] = pd.read_csv(path_data+df_name+'.csv', sep = sep, index_col=index_col, low_memory=False).replace({'\r\n': '\n'}, regex=True)

def import_dfs(df_names, path_data, sep = ',', index_col=None, verbose=False):
    for df_name in tqdm(df_names) : 
        if(verbose) :
            print(" - ", df_name)
        import_df(df_name, path_data, sep, index_col=index_col)

tables = [
    'identification_pz0',
    'entite_unique_par_sdc_nettoyage',
    'zone',
    'parcelle',
    'synthetise',
    'sdc'
]

# import des données du magasin
import_dfs(tables, ENTREPOT_PATH, sep = ',', verbose=False)

100%|██████████| 6/6 [00:06<00:00,  1.16s/it]


In [ ]:
# checking 1 
a = len(
    df.loc[(df['approche_de_calcul'] == 'synthétisé') &
        ~(df.apply(lambda row: str(row['campagne']) in str(row['synthetise_campagne']).split(', '), axis=1)),
        ['campagne','synthetise_campagne']]
)
print("nombre de synthétisé dont la campagne du domaine n'est pas dans les campagnes synthétisées :", a)
if a != 0: raise ValueError("Il reste des synthétisés dont la campagne du domaine n'est pas dans les campagnes synthétisées.")
else : print("Ok !")

In [ ]:
# Définir les années limites, soit celle en cours et la plus vieille acceptable 2008
annee_en_cours = datetime.now().year
annees_trop_vieille = 2004 # des pz0 attendues jusqu'en 2005

# Créer la fonction qui filtre les années au seins d'un synthétisé mutli-annuel
def filtrer_annees(annees_str):
    if pd.isna(annees_str):
        return None
    if not isinstance(annees_str, str): annees_str = str(annees_str)
    annees = [int(a.strip()) for a in annees_str.split(",")]
    annees_filtrees = [str(a) for a in annees if (a < annee_en_cours) & (a > annees_trop_vieille)]
    return ", ".join(annees_filtrees) if annees_filtrees else None

# Appliquer la fonction de filtrage des années aux campagnes en synthétisé
df['synthetise_campagne'] = df['synthetise_campagne'].apply(filtrer_annees)

# Si toutes les années d'un synthétisé ont été filtrées, on supprime la ligne
# Pareil pour les réalisés dont la campagne est trop récente ou trop vieille
df = df.loc[(pd.to_numeric(df['campagne'], errors='coerce') < annee_en_cours) &
            (pd.to_numeric(df['campagne'], errors='coerce') > annees_trop_vieille) &
            (df['synthetise_campagne'].notna())]

In [ ]:
def list_to_scalar(serie):
    unique_values = list(serie.dropna().unique())
    if len(unique_values) == 0:
        return None
    elif len(unique_values) == 1:
        return unique_values[0]
    else:
        return unique_values

# Préparer un dataframe des sdc réalisé avec leur pz0 identifié
pz0ident_real = intervention_realise_agrege.merge(identification_pz0.rename(columns={'entite_id':'zone_id'}), on='zone_id', how='left')[['sdc_id','pz0']].groupby('sdc_id')['pz0'].apply(lambda x: list_to_scalar(x), include_groups=False).reset_index()
pz0ident_real = pz0ident_real.loc[pz0ident_real['pz0'].notna()]

if len(pz0ident_real.loc[pz0ident_real['pz0'].apply(lambda x: isinstance(x, list))] ) > 0 :
    raise ValueError("Il y a des sdc réalisé avec plusieurs identification différentes selon leur zones")

# On supprime les synthétisés ou réalisés n'ayant pas de pz0 identifié correctement après avoir merge les identifications
df = df.merge(identification_pz0.rename(columns={'entite_id':'synthetise_id'}), on='synthetise_id', how='left')
df = df.merge(pz0ident_real.rename(columns={'entite_id':'sdc_id'}), on='sdc_id', how='left')
df['pz0'] = df['pz0_x'].combine_first(df['pz0_y'])
df = df.drop(columns=['pz0_x','pz0_y'])

print(f"Il y a {len(df.loc[~df['pz0'].isin(['pz0','post'])])} synthétisé ou sdc filtrés n'ayant pas de pz0 identifié correctement.")

df = df.loc[df['pz0'].isin(['pz0','post'])]